<a href="https://colab.research.google.com/github/KhushiShrivastav06/AI-Lab/blob/main/Week7_FirstOrderLogic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from itertools import product


def is_variable(term):
    """Variables start with '?' (e.g., '?x', '?y')."""
    return isinstance(term, str) and term.startswith("?")


def parse_atom(atom_str):
    """Parses 'Parent(Alice, ?x)' into ('Parent', ['Alice', '?x'])."""
    pred, args = atom_str.strip().rstrip(")").split("(")
    arg_list = [arg.strip() for arg in args.split(",")]
    return (pred.strip(), arg_list)


def format_atom(atom):
    """Formats ('Parent', ['Alice', 'Bob']) back into 'Parent(Alice, Bob)'."""
    pred, args = atom
    return f"{pred}({', '.join(args)})"


def substitute(atom, theta):
    """Applies variable bindings (theta) to an atom."""
    pred, args = atom
    new_args = []
    for arg in args:
        while arg in theta:
            arg = theta[arg]
        new_args.append(arg)
    return (pred, new_args)


def unify(x, y, theta=None):
    """Unifies two terms, argument lists, or atoms, returning a substitution dict."""
    if theta is None:
        theta = {}
    if theta is None:
        return None
    if x == y:
        return theta
    if is_variable(x):
        return unify_var(x, y, theta)
    if is_variable(y):
        return unify_var(y, x, theta)
    if isinstance(x, tuple) and isinstance(y, tuple):
        if x[0] != y[0] or len(x[1]) != len(y[1]):
            return None
        return unify(x[1], y[1], theta)
    if isinstance(x, list) and isinstance(y, list) and len(x) == len(y):
        for a, b in zip(x, y):
            theta = unify(a, b, theta)
            if theta is None:
                return None
        return theta
    return None


def unify_var(var, val, theta):
    if var in theta:
        return unify(theta[var], val, theta)
    if val in theta:
        return unify(var, theta[val], theta)
    # Occurs check: prevent infinite loops like ?x = f(?x)
    if isinstance(val, (list, tuple)) and var in str(val):
        return None
    new_theta = dict(theta)
    new_theta[var] = val
    return new_theta


def standardize_variables(rule, counter):
    """Renames variables in a rule so they don't clash across iterations."""
    premises, conclusion = rule
    mapping = {}

    def rename_atom(atom):
        pred, args = atom
        new_args = []
        for arg in args:
            if is_variable(arg):
                if arg not in mapping:
                    mapping[arg] = f"{arg}_{counter}"
                new_args.append(mapping[arg])
            else:
                new_args.append(arg)
        return (pred, new_args)

    return ([rename_atom(p) for p in premises], rename_atom(conclusion))


def fol_forward_chaining(facts, rules, query):
    """Derives new facts using Forward Chaining until the query is proven or no new facts emerge."""
    known_facts = set(facts)
    rule_counter = 0

    while True:
        new_facts = set()

        for rule in rules:
            rule_counter += 1
            premises, conclusion = standardize_variables(rule, rule_counter)

            # Find all facts matching each premise
            matching_facts_per_premise = [
                [f for f in known_facts if f[0] == p[0]] for p in premises
            ]

            # Try all combinations of candidate facts for the rule's premises
            for fact_combo in product(*matching_facts_per_premise):
                theta = {}
                for premise, fact in zip(premises, fact_combo):
                    theta = unify(premise, fact, theta)
                    if theta is None:
                        break

                if theta is not None:
                    inferred = substitute(conclusion, theta)
                    # Convert list args to tuple so the atom is hashable
                    inferred_hashable = (inferred[0], tuple(inferred[1]))
                    if inferred_hashable not in known_facts:
                        new_facts.add(inferred_hashable)
                        if unify(inferred, query, {}) is not None:
                            known_facts.update(new_facts)
                            return True, known_facts

        if not new_facts:
            return False, known_facts
        known_facts.update(new_facts)


if __name__ == "__main__":
    # 1. Base Facts
    raw_facts = [
        "Parent(Alice, Bob)",
        "Parent(Bob, Charlie)",
        "Parent(Charlie, Diana)",
        "Male(Bob)",
    ]
    facts = [(p, tuple(args)) for p, args in map(parse_atom, raw_facts)]

    # 2. Inference Rules: ([Premises], Conclusion)
    rules = [
        # Parent(?x, ?y) -> Ancestor(?x, ?y)
        ([parse_atom("Parent(?x, ?y)")], parse_atom("Ancestor(?x, ?y)")),
        # Parent(?x, ?y) ^ Ancestor(?y, ?z) -> Ancestor(?x, ?z)
        (
            [parse_atom("Parent(?x, ?y)"), parse_atom("Ancestor(?y, ?z)")],
            parse_atom("Ancestor(?x, ?z)"),
        ),
        # Parent(?x, ?y) ^ Male(?x) -> Father(?x, ?y)
        (
            [parse_atom("Parent(?x, ?y)"), parse_atom("Male(?x)")],
            parse_atom("Father(?x, ?y)"),
        ),
    ]

    # 3. Query the Knowledge Base
    query = parse_atom("Ancestor(Alice, Diana)")
    proven, all_facts = fol_forward_chaining(facts, rules, query)

    print(f"Query: {format_atom(query)} -> Proven: {proven}\n")
    print("All Known & Derived Facts:")
    for fact in sorted(all_facts):
        print(" -", format_atom((fact[0], list(fact[1]))))

Query: Ancestor(Alice, Diana) -> Proven: False

All Known & Derived Facts:
 - Male(Bob)
 - Parent(Alice, Bob)
 - Parent(Bob, Charlie)
 - Parent(Charlie, Diana)
